# Vérification réelle Qwen + LoRA sur GPU Colab
Choisir **Exécution → Modifier le type d'exécution → GPU**. Téléverser le projet **avec les modifications locales validées** (les scripts ne sont pas encore sur le HEAD distant) et les adaptateurs, ou monter un dossier Drive. Ne pas confondre un repli OCR avec une génération réelle.
Ce notebook n'a pas été exécuté dans l'environnement de développement. Aucun token ne doit être inscrit dans le notebook ; utiliser un secret Colab/Hugging Face si nécessaire.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
PROJECT = Path('/content/llm-extraction-finetune')
ADAPTER = Path('/content/lora_adapter')
ALLOW_DOWNLOAD = False  # Passer explicitement à True pour autoriser le Hub sur Colab.
STRESS = True
assert (PROJECT / 'scripts/run_real_model_check.py').is_file(), 'Téléverser la version locale complète du projet.'
os.chdir(PROJECT)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'tesseract-ocr', 'tesseract-ocr-fra', 'tesseract-ocr-eng'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-inference.txt'], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'Activer un GPU Colab.'
print(torch.cuda.get_device_name(0))
if not (ADAPTER / 'adapter_model.safetensors').is_file():
    assert ALLOW_DOWNLOAD, 'Téléverser les adaptateurs ou autoriser explicitement le téléchargement.'
    from huggingface_hub import snapshot_download
    snapshot_download('nicolbl95/llm-extraction-weights', local_dir=ADAPTER)
assert (ADAPTER / 'adapter_config.json').is_file()
assert (ADAPTER / 'adapter_model.safetensors').is_file()
assert (ADAPTER / 'tokenizer_config.json').is_file()


In [ ]:
from datetime import datetime, timezone
OUTPUT = PROJECT / 'results' / ('real-model-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.json')
cmd = [sys.executable, 'scripts/run_real_model_check.py', '--adapter-path', str(ADAPTER), '--output', str(OUTPUT), '--load-in-4bit']
if STRESS:
    cmd.append('--stress')
if ALLOW_DOWNLOAD:
    cmd.append('--allow-download')
completed = subprocess.run(cmd, check=False)
print('Code de retour :', completed.returncode, '(0 = tous les critères satisfaits ; un repli OCR seul ne suffit pas)')
assert OUTPUT.is_file(), 'Consulter les logs : le modèle n’a pas pu être chargé ou le contrôle a échoué avant exécution.'


In [ ]:
import pandas as pd
report = json.loads(OUTPUT.read_text())
table = []
for image in report['images']:
    runs = image['runs']
    table.append({'image': image['image'], 'modèle réel terminé': image['actual_model_completed'], 'référence exacte (3 fois)': all(image['final_equals_reference']), 'brut déterministe': image['raw_deterministic'], 'final déterministe': image['final_deterministic'], 'null à tort (somme 3 fois)': sum(run.get('false_nulls', 0) for run in runs), 'valeurs fausses conservées': sum(len(run.get('wrong_values_retained', [])) for run in runs), 'durée (s)': round(sum(run['duration_seconds'] for run in runs), 2)})
display(pd.DataFrame(table))
print('Résultats bruts, champs neutralisés, erreurs techniques et référence :', OUTPUT)
print('Une ligne sans génération réelle est NON VÉRIFIÉE côté modèle, même si le JSON OCR est exact.')
